# 02 — Reti di testate

Due testate sono **parenti** se c'è una prova che le gestisce la stessa organizzazione o che condividono
contenuti. Le testate parenti devono stare dalla stessa parte (tutte nel training o tutte nel test),
altrimenti il classificatore impara a riconoscere la rete invece che l'affidabilità.

Nel tentativo precedente (`EsperimentiPreliminari/2-NuovoScrapingConPagine`) si usavano solo gli articoli
quasi uguali: le reti che pubblicano articoli diversi con lo stesso template (FiveZero Media, i giornali
Carpenter Media Group…) restavano spezzate. Qui si usano cinque segnali, tutti **ciechi alle etichette e al
campo Owner di NewsGuard**:

| Segnale | Prova di parentela | Da dove |
|---|---|---|
| copie | articoli in comune (copie trovate in `01`): almeno `MIN_COPIE` e almeno `QUOTA_RELAZIONE` degli articoli della testata più piccola | testo |
| codici di account | stesso ID di AdSense, Google Analytics, Tag Manager, Facebook (pixel, app), Chartbeat, Quantcast, Comscore | HTML |
| titolare legale | la privacy policy o i termini d'uso sono la **stessa pagina** su un altro dominio (il titolare del trattamento dei dati) | HTML |
| dominio | stesso dominio registrato (sottodomini), o la homepage reindirizza allo stesso dominio | indirizzi |
| autori | gli stessi autori (nome e cognome) firmano almeno `QUOTA_RELAZIONE` degli articoli della testata più piccola | metadati degli articoli |

Per i due segnali di testo serve una relazione **sistematica**: un pezzo d'agenzia ripreso o un freelance che
scrive per due testate non bastano (legherebbero testate che non hanno niente in comune e creerebbero reti
giganti per catena). I segnali HTML, legali e di dominio sono prove a livello di account e bastano da soli.

Le reti sono le componenti connesse del grafo con un arco per ogni coppia con almeno un segnale. Il campo Owner
di NewsGuard serve **solo alla fine**, per misurare quanto le reti trovate coincidono con le proprietà note.

**Sul server** (serve l'HTML grezzo, che sta solo lì):

```bash
cd ~/John/WebScience2027/Versione2/1-data-cleaning
screen -dmS cleaning papermill -k johnvenv 02_reti.ipynb ../Results/1-data-cleaning/esecuzioni/02_reti.ipynb
```

In [1]:
import collections
import gzip
import itertools
import json
import os
import re
from urllib.parse import urljoin, urlparse

import networkx as nx
import numpy as np
import pandas as pd
import requests
import tldextract
from joblib import Parallel, delayed
from lxml import html as LH

from comune import DERIVED, GREZZI, NEWSGUARD, RESULTS, carica_articoli, normalizza_owner

os.makedirs(RESULTS, exist_ok=True)
dominio_registrato = lambda u: tldextract.extract(u).top_domain_under_public_suffix

In [2]:
MIN_COPIE = 2                  # un solo articolo in comune può essere un pezzo d'agenzia ripreso da tutti
QUOTA_RELAZIONE = 0.10         # copie e autori: almeno il 10% degli articoli della testata più piccola
PAGINE_HTML = 30               # pagine di articolo lette per testata
QUOTA_PAGINE_CODICE = 0.5      # un codice conta se è in almeno metà delle pagine (non in un widget occasionale)
QUOTA_PAGINE_LEGALE = 0.8      # il link legale è nel piè di pagina: deve esserci in quasi tutte le pagine
MIN_ARTICOLI_AUTORE = 2
N_JOBS = 32

In [3]:
art = carica_articoli().drop(columns=["label", "score"])      # cieco alle etichette
testate = sorted(art.domain.unique())
collegamenti = collections.defaultdict(set)                  # coppia di testate -> segnali


def collega(coppie, segnale):
    n = 0
    for a, b in coppie:
        if a != b:
            collegamenti[frozenset((a, b))].add(segnale); n += 1
    print(f"{segnale}: {n} coppie")


print(len(testate), "testate (tutte quelle dello scraping: una testata scartata dopo può comunque fare da ponte)")

1046 testate (tutte quelle dello scraping: una testata scartata dopo può comunque fare da ponte)


## 1. Copie di articoli

In [4]:
copie = pd.read_csv(DERIVED + "01_copie_tra_testate.csv")
n_copie = collections.Counter(frozenset(p) for p in zip(copie.testata_a, copie.testata_b))
print("coppie di testate per numero di articoli in comune:", collections.Counter(min(n, 10) for n in n_copie.values()))
n_articoli = art.groupby("domain").size()
sistematica = lambda p, n: n >= QUOTA_RELAZIONE * min(n_articoli[d] for d in p)
collega([tuple(p) for p, n in n_copie.items() if n >= MIN_COPIE and sistematica(p, n)], "copie")

coppie di testate per numero di articoli in comune: Counter({1: 5422, 10: 1884, 2: 1774, 3: 859, 4: 505, 5: 380, 6: 211, 7: 146, 8: 128, 9: 73})
copie: 4543 coppie


## 2. Codici di account nell'HTML

Ogni sito che usa AdSense, Google Analytics o il pixel di Facebook incorpora nelle pagine l'identificativo
del **proprio account**. Due siti con lo stesso identificativo sono gestiti dallo stesso account. Un codice
conta per una testata solo se è in almeno metà delle sue pagine.

In [5]:
CODICI = {
    "adsense": r"\bca-pub-\d{10,20}\b", "analytics": r"\bUA-\d{4,10}-\d{1,3}\b|\bG-[A-Z0-9]{8,12}\b",
    "tag_manager": r"\bGTM-[A-Z0-9]{4,9}\b", "facebook_pixel": r"fbq\(\s*['\"]init['\"]\s*,\s*['\"](\d{8,20})",
    "facebook_app": r"fb:app_id['\"]?\s+content=['\"](\d+)", "chartbeat": r"_sf_async_config\.uid\s*=\s*(\d+)",
    "quantcast": r"qacct\s*[:=]\s*['\"](p-[A-Za-z0-9_-]+)", "comscore": r"\bc2[\"']?\s*[:=]\s*[\"'](\d{6,9})",
}
LEGALE = re.compile(r"privacy|terms of (use|service)|do not sell|cookie policy|terms (&|and) conditions", re.I)


def pagine(d):
    f = f"{GREZZI}html/{d}.jsonl.gz"
    if not os.path.exists(f):
        return
    n = 0
    for riga in gzip.open(f, "rt"):
        r = json.loads(riga)
        if r.get("tipo") == "articolo" and r.get("html"):
            yield r
            n += 1
            if n >= PAGINE_HTML:
                return


def impronte_html(d):
    """Codici di account e link legali esterni presenti nelle pagine di articolo della testata."""
    codici, legali, n = collections.Counter(), collections.Counter(), 0
    for r in pagine(d):
        n += 1
        h = r["html"]
        codici.update({f"{k}:{m}" for k, p in CODICI.items() for m in set(re.findall(p, h))})
        try:
            albero = LH.fromstring(h)
        except Exception:
            continue
        casa, visti = dominio_registrato(r["url"]), set()
        for a in albero.xpath("//a[@href]"):
            if LEGALE.search(a.text_content() or ""):
                u = urljoin(r["url"], a.get("href").strip())
                if dominio_registrato(u) and dominio_registrato(u) != casa:
                    p = urlparse(u)
                    visti.add((p.netloc.lower().removeprefix("www.") + p.path.rstrip("/")).lower())
        legali.update(visti)
    return {"testata": d, "pagine": n,
            "codici": [k for k, v in codici.items() if n >= 2 and v >= QUOTA_PAGINE_CODICE * n],
            "legali": [k for k, v in legali.items() if n >= 5 and v >= QUOTA_PAGINE_LEGALE * n]}


impronte = pd.DataFrame(Parallel(n_jobs=N_JOBS)(delayed(impronte_html)(d) for d in testate)).set_index("testata")
print(f"testate con HTML: {(impronte.pagine > 0).sum()}, con almeno un codice: {(impronte.codici.str.len() > 0).sum()}")


def per_valore(colonna):
    v = collections.defaultdict(set)
    for d, lista in impronte[colonna].items():
        for x in lista:
            v[x].add(d)
    return {k: s for k, s in v.items() if len(s) >= 2}


codici_condivisi = per_valore("codici")
for tipo in CODICI:
    collega([p for k, s in codici_condivisi.items() if k.startswith(tipo + ":") for p in itertools.combinations(sorted(s), 2)],
            f"codice {tipo}")

testate con HTML: 1046, con almeno un codice: 542
codice adsense: 212 coppie
codice analytics: 42 coppie
codice tag_manager: 71 coppie
codice facebook_pixel: 25 coppie
codice facebook_app: 66 coppie
codice chartbeat: 23 coppie
codice quantcast: 0 coppie
codice comscore: 1 coppie


## 3. Titolare legale

Il link "Privacy Policy" (o "Terms of Use", "Do Not Sell My Personal Information") porta alla pagina che
dichiara chi è il titolare del trattamento dei dati. Se due siti puntano alla **stessa pagina** su un altro
dominio, hanno lo stesso titolare (per esempio tutti i giornali Carpenter Media Group puntano a
`carpentermediagroup.com/privacy-policy`).

Non contano le pagine dei servizi di terze parti e delle piattaforme che ospitano il sito (Google, Substack,
WordPress.com…): sono uguali per tutti i loro clienti. Per i servizi si usa la lista pubblica di Disconnect
(quella dei blocchi di tracciamento di Firefox); per le piattaforme di hosting e i servizi di newsletter che
la lista non copre, un elenco breve, qui sotto.

In [6]:
F_DISCONNECT = DERIVED + "disconnect_services.json"
if not os.path.exists(F_DISCONNECT):
    r = requests.get("https://raw.githubusercontent.com/disconnectme/disconnect-tracking-protection/master/services.json", timeout=60)
    open(F_DISCONNECT, "w").write(r.text)
servizi = set()
for lista in json.load(open(F_DISCONNECT))["categories"].values():
    for ente in lista:
        for sotto in ente.values():
            for url, domini in sotto.items():
                if isinstance(domini, list):
                    servizi |= {dominio_registrato("http://" + x) for x in domini} | {dominio_registrato(url)}
PIATTAFORME = {"substack.com", "medium.com", "wordpress.com", "blogspot.com", "blogger.com", "ghost.io",
               "squarespace.com", "wix.com", "weebly.com", "google.com", "tumblr.com",
               # servizi di newsletter e di conformità email (Constant Contact e simili), non nella lista Disconnect
               "safesubscribe.org", "safesubscribe.com", "constantcontact.com", "email-comply.com"}
esclusi = (servizi | PIATTAFORME) - {""}
legali_condivisi = {k: s for k, s in per_valore("legali").items() if dominio_registrato("http://" + k) not in esclusi}
for k, s in sorted(legali_condivisi.items(), key=lambda x: -len(x[1])):
    print(f"  {len(s):3d} testate -> {k}")
collega([p for s in legali_condivisi.values() for p in itertools.combinations(sorted(s), 2)], "titolare legale")

   36 testate -> carpentermediagroup.com/privacy-policy
   36 testate -> carpentermediagroup.com/terms-of-service
    8 testate -> salemmedia.com/privacy-policy
    5 testate -> privacywatch.news
    4 testate -> login.nine.com.au/privacy
    4 testate -> login.nine.com.au/terms
    4 testate -> cdn.townhall.com/web/privacypolicy/termsofuse.htm
    3 testate -> sevenwestmedia.com.au/privacy-policies
    3 testate -> newstarget.com/privacypolicy.html
    2 testate -> mailsubscriptions.co.uk/info/368160/subscriptions-general-terms-and-conditions
    2 testate -> foxnews.com/privacy-policy
    2 testate -> foxnews.com/terms-of-use
    2 testate -> iubenda.com/privacy-policy/18441249
    2 testate -> iubenda.com/privacy-policy/18441249/cookie-policy
    2 testate -> postmedia.com/terms-and-conditions
    2 testate -> corporate.televisaunivision.com/privacy-notice
titolare legale: 1329 coppie


## 4. Dominio

Sottodomini dello stesso dominio registrato (per esempio le edizioni locali di un sito), e siti la cui
homepage reindirizza allo stesso dominio.

In [7]:
registrato = {d: dominio_registrato(d) for d in testate}
casa = {}
for d in testate:
    try:
        casa[d] = dominio_registrato(json.load(open(f"{GREZZI}stato/{d}.json")).get("homepage", "")) or registrato[d]
    except FileNotFoundError:
        casa[d] = registrato[d]
for nome, chiave in [("stesso dominio registrato", registrato), ("redirect allo stesso dominio", casa)]:
    collega([p for _, g in itertools.groupby(sorted(testate, key=chiave.get), key=chiave.get)
             for p in itertools.combinations(list(g), 2)], f"dominio: {nome}")

dominio: stesso dominio registrato: 53 coppie
dominio: redirect allo stesso dominio: 102 coppie


## 5. Autori in comune

Il campo autore estratto dalla pagina. Contano solo nomi di persona (da 2 a 4 parole) che non sono firme
generiche (redazione, agenzie, "Staff"…), con almeno `MIN_ARTICOLI_AUTORE` articoli in ciascuna testata. Due
testate sono legate se gli autori in comune firmano almeno `QUOTA_RELAZIONE` degli articoli della più piccola
(per ogni autore conta il minimo dei suoi articoli nelle due testate): una redazione in comune, non un
collaboratore occasionale.

In [8]:
autore = art.autore.fillna("").str.lower().str.strip()
GENERICHE = re.compile(r"staff|editor|admin|desk|team|report|news|press|wire|contributor|guest|associated|reuters|^by$")
buoni = autore.str.split().str.len().between(2, 4) & ~autore.str.contains(GENERICHE)
firme = art[buoni].assign(autore=autore[buoni]).groupby(["autore", "domain"]).size()
firme = firme[firme >= MIN_ARTICOLI_AUTORE].reset_index()
in_comune = collections.Counter()
for _, g in firme.groupby("autore"):
    quanti = dict(zip(g.domain, g[0]))
    for a, b in itertools.combinations(sorted(quanti), 2):
        in_comune[frozenset((a, b))] += min(quanti[a], quanti[b])
collega([tuple(p) for p, n in in_comune.items() if sistematica(p, n)], "autori")

autori: 54968 coppie


## Le reti

Componenti connesse del grafo delle testate.

In [9]:
SEGNALI = sorted({s for v in collegamenti.values() for s in v})
FAMIGLIA = lambda s: s.split(":")[0].split(" ")[0]


def reti(con=None):
    G = nx.Graph(); G.add_nodes_from(testate)
    G.add_edges_from(tuple(p) for p, s in collegamenti.items() if con is None or s & con)
    return {d: i for i, c in enumerate(sorted(nx.connected_components(G), key=len, reverse=True)) for d in c}


rete = reti()
dim = collections.Counter(rete.values())
print(f"{len(dim)} reti; testate in una rete con altre: {sum(n for n in dim.values() if n > 1)}; "
      f"dimensione delle reti più grandi: {[n for _, n in dim.most_common(10)]}")

375 reti; testate in una rete con altre: 725; dimensione delle reti più grandi: [426, 42, 41, 39, 18, 11, 8, 8, 7, 6]


## Verifica con il campo Owner di NewsGuard

B-cubed sulle testate con un owner noto: **precisione** = quanto le testate della mia rete hanno il mio stesso
owner; **recall** = quanto le testate con il mio owner stanno nella mia rete. Un "errore" di precisione non è
sempre un errore: due testate di owner diversi possono condividere contenuti o gestione (lo si vede negli
esempi sotto), ed è proprio quello che vogliamo separare.

In [10]:
ng = pd.read_csv(NEWSGUARD, low_memory=False)
ng = ng.assign(d=ng.Domain.str.lower()).drop_duplicates("d").set_index("d")
owner = {d: normalizza_owner(ng.Owner.get(d, "")) for d in testate}
noti = [d for d in testate if owner[d]]


def bcubed(r):
    P = np.mean([np.mean([owner[e] == owner[d] for e in noti if r[e] == r[d]]) for d in noti])
    R = np.mean([np.mean([r[e] == r[d] for e in noti if owner[e] == owner[d]]) for d in noti])
    return round(P, 3), round(R, 3)


famiglie = sorted({FAMIGLIA(s) for s in SEGNALI})
righe = {"nessun segnale (ogni testata da sola)": bcubed({d: d for d in testate})}
for f in famiglie:
    righe[f"solo {f}"] = bcubed(reti({s for s in SEGNALI if FAMIGLIA(s) == f}))
for f in famiglie:
    righe[f"tutti tranne {f}"] = bcubed(reti({s for s in SEGNALI if FAMIGLIA(s) != f}))
righe["tutti i segnali"] = bcubed(rete)
verifica = pd.DataFrame(righe, index=["precisione", "recall"]).T
verifica.to_csv(RESULTS + "02_verifica_owner.csv")
verifica

,precisione,recall
nessun segnale (ogni testata da sola),1.000,0.570
solo autori,0.982,0.693
solo codice,0.965,0.683
solo copie,0.975,0.725
solo dominio,0.991,0.630
solo titolare,0.976,0.624
tutti tranne autori,0.918,0.859
tutti tranne codice,0.937,0.827
tutti tranne copie,0.928,0.837
tutti tranne dominio,0.914,0.867


In [11]:
# le proprietà con più testate: in quante reti finiscono
righe = []
for o, g in itertools.groupby(sorted(noti, key=owner.get), key=owner.get):
    g = list(g)
    if len(g) >= 3:
        c = collections.Counter(rete[d] for d in g)
        righe.append({"owner": o, "testate": len(g), "reti": len(c), "nella rete principale": c.most_common(1)[0][1]})
pd.DataFrame(righe).sort_values("testate", ascending=False).set_index("owner")

,testate,reti,nella rete principale
owner,,,
boone newspapers,54,20,30
national world,45,5,41
newsquest,38,1,38
fivezero,18,1,18
retirement systems of alabama,12,3,9
fox,11,1,11
salem,9,2,8
buzzfeed and verizon verizon communications,8,1,8
russian government,6,2,3


In [12]:
# le reti più grandi: da quali owner sono fatte, e i collegamenti tra owner diversi (da guardare a mano)
for r_id, n in dim.most_common(8):
    membri = [d for d in testate if rete[d] == r_id]
    print(f"\nrete {r_id} ({n} testate): owner {dict(collections.Counter(owner[d] or '?' for d in membri).most_common(5))}")
    ponti = [(sorted(p), s) for p, s in collegamenti.items() if p <= set(membri) and len({owner[d] for d in p}) == 2 and all(owner[d] for d in p)]
    for (a, b), s in ponti[:8]:
        print(f"   {a} ({owner[a][:20]}) — {b} ({owner[b][:20]}): {', '.join(sorted(s))}")


rete 0 (426 testate): owner {'?': 426}



rete 1 (42 testate): owner {'newsquest': 38, 'gannett': 2, 'national world': 1, 'evening standard evgeny lebedev sultan mohamed abuljadayel daily mail and general justin byam shaw geordie greig': 1}
   barkinganddagenhampost.co.uk (newsquest) — dorsetecho.co.uk (gannett): copie
   barkinganddagenhampost.co.uk (newsquest) — leighjournal.co.uk (national world): copie
   becclesandbungayjournal.co.uk (newsquest) — dorsetecho.co.uk (gannett): copie
   becclesandbungayjournal.co.uk (newsquest) — leighjournal.co.uk (national world): copie
   becclesandbungayjournal.co.uk (newsquest) — standard.co.uk (evening standard evg): copie
   cambstimes.co.uk (newsquest) — dorsetecho.co.uk (gannett): copie
   cambstimes.co.uk (newsquest) — leighjournal.co.uk (national world): copie
   cambstimes.co.uk (newsquest) — standard.co.uk (evening standard evg): copie

rete 2 (41 testate): owner {'national world': 41}

rete 3 (39 testate): owner {'boone newspapers': 30, 'retirement systems of alabama': 9}
   a

In [13]:
pd.DataFrame({"domain": testate, "rete": [rete[d] for d in testate], "testate_nella_rete": [dim[rete[d]] for d in testate]}).to_csv(
    RESULTS + "02_reti.csv", index=False)
pd.DataFrame([{"testata_a": a, "testata_b": b, "segnali": "; ".join(sorted(s))} for p, s in collegamenti.items()
              for a, b in [sorted(p)]]).to_csv(RESULTS + "02_collegamenti.csv", index=False)
print("salvati 02_reti.csv e 02_collegamenti.csv")

salvati 02_reti.csv e 02_collegamenti.csv
